# Pentora CART — Colab runner

Autonomous **Continuous Automated Red Teaming** on a local LLM. Nothing leaves this VM.

Run the cells top to bottom:
1. Install the engine (from the `feature/engine-core` branch)
2. Boot a local Ollama + pull the model
3. **Self-contained demo** — the engine proves a real IDOR against a vulnerable API spun up inside this VM (no external target, fully legal)
4. Template to scan **your own authorized target** from a HAR export

> Only test systems you own or are authorized to test.

## 1. Install the engine

In [ ]:
# 1) Install the CART engine from the branch (NOT PyPI — the engine isn't released yet)
!pip -q install "git+https://github.com/sohan-a11y/pentora.git@feature/engine-core#egg=pentora[engine,capture]"
print("engine installed")


## 2. Local Ollama (optional)

In [ ]:
# 2) Install + boot a local Ollama and pull the model (OPTIONAL — the deterministic
#    playbooks in cell 3 work without it; the LLM only adds smarter 'where to look' routing).
import os, time, shutil, subprocess, urllib.request
!curl -fsSL https://ollama.com/install.sh | sh
ollama = shutil.which("ollama") or "/usr/local/bin/ollama"   # install drops it here
print("ollama binary:", ollama)
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
# start the server as a background child of this kernel so it survives across cells
subprocess.Popen([ollama, "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
# wait until the API actually answers (don't guess with a fixed sleep)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        print("ollama server is up"); break
    except Exception:
        time.sleep(2)
else:
    print("ollama did NOT come up — tail of /tmp/ollama.log:")
    print(open("/tmp/ollama.log").read()[-1500:])
subprocess.run([ollama, "pull", "qwen3:8b"], check=False)   # ~5 GB, a minute or two on first run
subprocess.run([ollama, "ps"], check=False)                # should show the model on 100% GPU
print("done — the T4 is used automatically; cell 3 runs even if this was skipped")


## 3. Self-contained demo — prove a finding with zero external dependencies

In [ ]:
# 3) SELF-CONTAINED DEMO — no external target needed, fully legal.
# Spins a deliberately-vulnerable JWT/IDOR API inside this VM and lets the engine PROVE an
# IDOR: as user_b, request user_a's order and confirm user_a's private data comes back.
import base64, hashlib, hmac, json, threading
from http.server import BaseHTTPRequestHandler, HTTPServer

from pentora.engine import (
    Blackboard, DeterministicValidator, Governor, Hypothesis, RunScope,
)
from pentora.engine.playbook_idor import IdorPlaybookContext, run_idor_playbook

SECRET = "s3cr3t"
ORDERS = {"1": ("user_a", "ALPHA-secret-order-one"), "2": ("user_b", "BETA-secret-order-two")}

def b64u(b): return base64.urlsafe_b64encode(b).rstrip(b"=").decode()

def make_jwt(sub):
    h = b64u(json.dumps({"alg": "HS256", "typ": "JWT"}).encode())
    p = b64u(json.dumps({"sub": sub, "role": "user"}, separators=(",", ":")).encode())
    sig = b64u(hmac.new(SECRET.encode(), f"{h}.{p}".encode(), hashlib.sha256).digest())
    return f"{h}.{p}.{sig}"

def token_sub(token):
    try:
        h, p, sig = token.split(".")
        exp = b64u(hmac.new(SECRET.encode(), f"{h}.{p}".encode(), hashlib.sha256).digest())
        if not hmac.compare_digest(exp, sig): return None
        return str(json.loads(base64.urlsafe_b64decode(p + "=" * (-len(p) % 4)))["sub"])
    except Exception:
        return None

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        oid = self.path.rstrip("/").rsplit("/", 1)[-1]
        auth = self.headers.get("Authorization", "")
        sub = token_sub(auth[7:].strip() if auth[:7].lower() == "bearer " else "")
        if sub is None:
            code, body = 401, "unauthorized"
        elif oid not in ORDERS:
            code, body = 404, "not found"
        else:
            # VULNERABLE: returns any order by id, ignoring ownership
            code, body = 200, ORDERS[oid][1]
        data = body.encode()
        self.send_response(code); self.send_header("Content-Length", str(len(data))); self.end_headers()
        self.wfile.write(data)
    def log_message(self, *a): pass

srv = HTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
base = f"http://127.0.0.1:{srv.server_address[1]}"
print("vulnerable demo API listening at", base)

bb = Blackboard()
hyp = bb.assert_fact(Hypothesis(source="demo", claim="idor"))
status = run_idor_playbook(IdorPlaybookContext(
    bb=bb, governor=Governor(), validator=DeterministicValidator(), hypothesis=hyp,
    victim_url=f"{base}/api/orders/1", victim_token=make_jwt("user_a"),
    attacker_token=make_jwt("user_b"), attacker_own_url=f"{base}/api/orders/2",
    scope=RunScope(include=["127.0.0.1"], read_only=True),
))
print("playbook status:", status)
for f in bb.query("finding"):
    print(f"\nFINDING: {f.title}  CVSS {f.cvss_score} ({f.severity})")
    print("  evidence:", f.evidence)
    print("  PoC:", f.poc)
if not bb.query("finding"):
    print("no finding (unexpected for this vulnerable demo)")


## 4. Scan your own authorized target (HAR upload)

In [ ]:
# 4) SCAN YOUR OWN AUTHORIZED TARGET
# Capture traffic in your browser (DevTools -> Network -> Save all as HAR), upload it here,
# then run the full autonomous engine.  ONLY scan systems you own or are authorized to test.
from google.colab import files
from pentora.engine.app import start

TARGET = "https://your-app.example.com"     # <-- edit me (authorized target only)

up = files.upload()                          # pick your traffic.har
har_path = next(iter(up))

e = start(target=TARGET, model="qwen3:8b", scope_hosts=[TARGET.split("//")[-1].split("/")[0]])
e.ingest_har(har_path)
summary = e.run()
print(summary)
print(e.report_markdown("pentora-cart-report.md"))

# Continuous mode: save today's findings as the baseline, then later diff to see only NEW ones.
# e.save_baseline("baseline.json")
